In [1]:
%run Hilbert_C^2_package.ipynb
# 标准库
import os
import time
import math
import csv
import json
import datetime
from itertools import combinations
from math import ceil

# 第三方库
import numpy as np
import resource

# Sage 相关
from sage.rings.polynomial.polydict import ETuple

# 2. 定义一元多项式环：变量为p，系数域为有理数域Q
Guess_R.<pp> = PolynomialRing(QQ)

In [2]:
# 切换第二个term需要改这个代码块

def find_valid_tuples_general(power_pattern, max_index):
    
    #核心函数，如果后面还要更大规模可以让豆包优化一下
    
    """
    【超级通用版】寻找满足次数约束的元组 (w2,w3,w4,...)
    输入：
        power_pattern: 幂次规则，字典形式
            例1：{1:2, 'w2':1, 'w3':1, 'w4':1} → s1² s_w2 s_w3 s_w4
            例2：{1:1, 'w2':2, 'w3':1, 'w4':1} → s1 s_w2² s_w3 s_w4
            例3：{1:2, 'w2':1, 'w3':1, 'w4':1, 'w5':1} → 扩展到 w5
        max_index: 最大下标，如 4 → 到 w4；5 → 到 w5
    输出：
        所有合法元组 (w2,w3,w4,...)
    """
    from itertools import product

    # 自动生成变量列表 [w2,w3,...wmax]
    var_list = [f'w{i}' for i in range(2, max_index+1)]
    
    # 自动生成每个 w_i 的范围：1<=w_i<=i
    ranges = [range(1, i+1) for i in range(2, max_index+1)]
    
    valid = []
    
    # 遍历所有可能组合
    for values in product(*ranges):
        # 映射：w2→v2, w3→v3,...
        val_map = {var: val for var, val in zip(var_list, values)}
        
        # 初始化次数
        deg = {}
        
        # 统计每一项的次数
        for key, p in power_pattern.items():
            if key == 1:
                # 固定项 s1
                j = 1
            else:
                # 变量项 sw_i → 取当前值
                j = val_map[key]
            
            if j in deg:
                deg[j] += p
            else:
                deg[j] = p
        
        # 检查：所有次数 ≤ 2
        if all(d <= 2 for d in deg.values()):
            valid.append(values)
    
    return valid

In [3]:
# 切换第二个term需要改这个代码块

def get_u(pattern, valid_tuple):
    max_index = len(valid_tuple) + 2
    res = [2] * max_index
    
    for key, val in pattern.items():
        if key == 1:
            # 纯数字 key：1 → 位置 1
            idx = key - 1
        else:
            # 字符串 key：'w2' → 提取数字 2
            idx = valid_tuple[int(key[1:]) - 2] - 1
        
        # 安全判断：不越界
        if 0 <= idx <= max_index:
            res[idx] -= val
    
    return res

In [4]:
def build_weight_matrix(w_tuple):
    """
    生成你要的 k×k 矩阵：第j列 wj行~j行=1，其余=0
    """
    k = len(w_tuple)
    M = matrix(ZZ, k, k)  # 整数环零矩阵
    
    for j in range(1, k+1):
        wj = w_tuple[j-1]
        start = wj
        end = j
        for i in range(start, end+1):
            M[i-1, j-1] = 1
    return M


def compute_inverse_times_vector(w_tuple):
    """
    主函数：
    1. 生成矩阵
    2. 求逆
    3. 生成向量 (d2+...+d_{k+1}, ..., d_{k+1})
    4. 返回 逆矩阵 × 向量
    """
    k = len(w_tuple)
    M = build_weight_matrix(w_tuple)
    
    # 求逆矩阵（Sage自动计算）
    M_inv = M.inverse()
    
    # 定义符号 d2, d3, ..., d_{k+1}
    vars_list = [var(f'd{i}') for i in range(2, k+2)]  # [d2,d3,...,d_{k+1}]
    
    # 生成你要的向量：(d2+...+d_{k+1}, ..., d_k+d_{k+1}, d_{k+1})
    vec = []
    for i in range(k):
        # 第i项：d_{2+i} + ... + d_{k+1}
        term = sum(vars_list[i:])
        vec.append(term)
    vec = vector(vec)
    
    # 计算 逆矩阵 × 向量
    result = M_inv * vec
    
    return [M, M_inv, vec, result]

In [5]:
def get_r_deg(u_tuple):
    k = len(u_tuple)
    var_list = [var(f'r{i}') for i in range(1, k+1)]
    r_vec = [0] * k
    r_vec[0] = sum(var_list[:u_tuple[0]])
    for i in range(1,k):
        r_vec[i] = sum(var_list[:sum(u_tuple[:i+1])]) - sum(var_list[:sum(u_tuple[:i])])
    return r_vec

def compute_final_vector(w_tuple, u_tuple):
    """
    主函数：
    1. 生成矩阵
    2. 求逆
    3. 生成向量 (d2+...+d_{k+1}, ..., d_{k+1})
    4. 返回 逆矩阵 × 向量
    """
    k = len(w_tuple) + 1
    M = build_weight_matrix((1,) + w_tuple)
    # 求逆矩阵（Sage自动计算）
    M_inv = M.inverse()
    
    # 定义deg (r1+,..., ..+r_k)
    r_list = get_r_deg(u_tuple)
    
    # 生成你要的向量：(d2+...+d_{k+1}, ..., d_k+d_{k+1}, d_{k+1})
    vec = [sum(r_list[i+1:]) for i in range(k)]
    vec = vector(vec)
    #print(vec)
    # 计算 逆矩阵 × 向量
    result = M_inv * vec
    
    return result

In [6]:
from itertools import permutations

def generate_sequences(p):
    """
    根据 p 数组生成所有符合要求的序列
    :param p: 列表，p[i] 表示数字 (i+1) 出现的次数
    :return: 所有不重复的排列
    """
    # 1. 构建基础数字列表：数字 i 出现 p[i-1] 次
    elements = []
    for num, count in enumerate(p, start=1):
        elements.extend([num] * count)
    
    # 2. 生成所有排列（自动去重）
    unique_perms = sorted(set(permutations(elements)))
    
    return unique_perms

# ------------------- 测试：你给的例子 -------------------

# 你的示例：l=4，p = [0,0,2,2]
p = [0, 0, 2, 2]

# 生成结果
result = generate_sequences(p)
    
# 输出
print("p =", p, "对应的所有排列：")
for seq in result:
    print(seq)
    
print(f"\n总计 {len(result)} 种排列方式")

p = [0, 0, 2, 2] 对应的所有排列：
(3, 3, 4, 4)
(3, 4, 3, 4)
(3, 4, 4, 3)
(4, 3, 3, 4)
(4, 3, 4, 3)
(4, 4, 3, 3)

总计 6 种排列方式


In [7]:
def merge_list(a, b):
    """
    Example:
    
    exm_a = [1, 2, 3, 4, 5, 6]
    exm_b = (3, 6, 5, 5, 6, 6)
    -----> [0, 0, 1, 0, 7, 13]
    """
    n = len(a)
    res = [0]*n
    for idx in range(n):
        res[int(b[idx]) - 1] += a[idx]
    return res

def compute_inverse_times_vector_with_value(M_inv, r_vec):
    """
    主函数：
    1. 生成矩阵
    2. 求逆
    3. 生成向量 (d2+...+d_{k+1}, ..., d_{k+1})
    4. 返回 逆矩阵 × 向量
    """
    k = len(r_vec) - 1
        
    # 定义符号 d2, d3, ..., d_{k+1}
    vars_list = r_vec[1:]  # [d2,d3,...,d_{k+1}]
    
    # 生成你要的向量：(d2+...+d_{k+1}, ..., d_k+d_{k+1}, d_{k+1})
    vec = []
    for i in range(k):
        # 第i项：d_{2+i} + ... + d_{k+1}
        term = sum(vars_list[i:])
        vec.append(term)
    vec = vector(vec)
    
    # 计算 逆矩阵 × 向量
    result = M_inv * vec
    
    return result

def build_dict(k, psi):
    if psi == 2:
        return {1: 2} | {f'w{i}': 1 for i in range(2, k)}
    else:
        return {1: 1} | {f'w{i}': 2 if i == psi-1 else 1 for i in range(2, k)}

In [8]:
def sigma(x):
    """
    自定义符号函数：
    - 输入x < 0 → 返回 1
    - 输入x = 0 或 x > 0 → 返回 0
    """
    if x < 0:
        return 1
    else:
        return 0

def cal_poly(par, r_length, r_deg):
    """
    计算公式 sum_{i<j<...<r_length} p^sigma(r_i + r_j + ... + r_{r_length}) * (r_i + r_j + ...+ r_{r_length})^{r_deg}
    其中：
    - r_i/r_j/r_k ∈ {1,2,3,4} 且 i<j<k
    - sigma函数：负数返回1，0/正数返回0
    返回关于p的一元多项式
    """
    total = 0
    # 生成所有满足i<j<k的三元组（1-4中选3个不重复的数）
    for r in combinations(par, r_length):
        s = sum(r)          # 计算(r_i + r_j + r_k)
        sigma_val = sigma(s)     # 计算sigma(r_i+r_j+r_k)
        term = (pp^sigma_val) * (s^r_deg)  # 计算每一项：p^sigma值 * 和的4次方
        total += term
        
    return total

In [9]:
'''par = [3, 2, 1, 1, -2, -2]
w_tuple = (1, 2, 2, 3, 3)
M = build_weight_matrix(w_tuple)
M_inv = M.inverse()
r_vec = merge_list(par, (5, 4, 6, 4, 6, 5))
res = compute_inverse_times_vector_with_value(M_inv, r_vec) # t_1, ..., t_(k-1)
print(res)'''

'par = [3, 2, 1, 1, -2, -2]\nw_tuple = (1, 2, 2, 3, 3)\nM = build_weight_matrix(w_tuple)\nM_inv = M.inverse()\nr_vec = merge_list(par, (5, 4, 6, 4, 6, 5))\nres = compute_inverse_times_vector_with_value(M_inv, r_vec) # t_1, ..., t_(k-1)\nprint(res)'

In [ ]:
#--------------
# 算例
# Basic settings
k = 6
n = 3
l_pos = 3
s_pos = 20

l_neg = k - l_pos
s_neg = s_pos + n
P_pos_list = Partitions(s_pos, length = l_pos)
P_neg_list = Partitions(s_neg, length = l_neg)
lam_pos = P_pos_list.random_element()
lam_neg = P_neg_list.random_element()
print(lam_neg,lam_pos)

In [ ]:
# 定义幂次模式
psi = 3 # range from 2 to k
pattern = build_dict(k, psi) #一切改动的根源，长度=k-1
print(pattern)
# 定义系数函数
coeff_pattern = lambda t_res: prod(t_res) * t_res[psi - 2] * pp^(sigma(t_res[psi-2])) #改完这里要改，长度=k-1

# 是否展示具体数据
if_print = False

w_list = find_valid_tuples_general(pattern, max_index=k-1)
if_print and print("合法元组：", w_list)
u_list = [get_u(pattern, word) for word in w_list]
if_print and print('u obeys: ', u_list)
deg_list = [compute_inverse_times_vector((1,) + word)[-1] for word in w_list]
fin_list = [compute_final_vector(w,u) for w, u in zip(w_list,u_list)]
for word in w_list:
    # 输入你的元组 (w1,w2,w3,w4)
    w_tuple = (1,) + word

    # 计算
    M, M_inv, vec, res = compute_inverse_times_vector(w_tuple)

    # 输出
    '''print("=== 生成的矩阵 ===")
    print(M)

    print("\n=== 逆矩阵 ===")
    print(M_inv)

    print("\n=== 你的向量 ===")
    print(vec)
    '''
    if if_print:
        print("\n=== 逆矩阵 × 向量 结果 ===")
        print(res)
if if_print:    
    for w,u in zip(w_list,u_list):
        print('\n===r_vec result is ===')
        print(compute_final_vector(w,u))

In [ ]:
gen_par = list(lam_neg) + [-i for i in lam_pos]

final_res = 0
for w_tuple, u_tuple in zip(w_list,u_list):
    M = build_weight_matrix((1,) + w_tuple)
    M_inv = M.inverse()
    for perm in generate_sequences(u_tuple):
        r_vec = merge_list(gen_par, perm)
        t_res = compute_inverse_times_vector_with_value(M_inv, r_vec) # t_1, ..., t_(k-1)
        cof_res = coeff_pattern(t_res)
        final_res += cof_res
print('=========final result of Coeff is =========')
print(final_res)
print(final_res.coefficients())
final_constant = final_res.coefficients()[0]
if len(final_res.coefficients()) == 2:
    final_coeff = final_res.coefficients()[-1]
else:
    final_coeff = 0
print(final_constant, final_coeff)

In [ ]:
component_list = []
for length in range(2,k):
    component_list += [cal_poly(gen_par, length, deg) * n^(k-deg) for deg in range(length+1, k+1)]

comp_0 = [p.coefficients()[0] for p in component_list]
comp_1 = [p.coefficients()[-1] if len(p.coefficients()) == 2 else 0 for p in component_list]

print(comp_0)
print(comp_1)

In [ ]:
A_coefficient_matrix.append(comp_0)
A_coefficient_matrix.append(comp_1)
b_vec.append(final_constant)
b_vec.append(final_coeff)
print(len(A_coefficient_matrix), len(b_vec))
for A_res in A_coefficient_matrix:
    print(A_res)
print("===============")
print(b_vec) # 长度对了再执行下一格

In [ ]:
size = (k-2) * (k-1) / 2
if len(b_vec) >= size:
    A_coefficient_matrix = np.array(A_coefficient_matrix[:size])
    b_vec = np.array(b_vec[:size])
    solution = [int(round(x)) for x in np.linalg.solve(A_coefficient_matrix, b_vec)]
    print(solution)
else:
    print("size is wrong")

In [ ]:
A_coefficient_matrix = []
b_vec = []

In [ ]:
"""======================="""

In [ ]:
from IPython.display import display, HTML

def show_clean_table(w_list, u_list, result_list, fin_list):
    html = '''
    <style>
        .custom-table {
            font-family: Arial, sans-serif;
            border-collapse: collapse;
            width: 100%;
            font-size: 14px;
        }
        .custom-table th, .custom-table td {
            border: 1px solid #ddd;
            padding: 8px 12px;
            text-align: left;
        }
        .custom-table th {
            background-color: #f5f5f5;
        }
    </style>
    <table class="custom-table">
        <tr>
            <th>w_tuple</th>
            <th>u_tuple</th>
            <th>result</th>
            <th>r_deg</th>
        </tr>
    '''
    
    for w, u, res, fin in zip(w_list, u_list, result_list, fin_list):
        html += f"<tr><td>{w}</td><td>{u}</td><td>{res}</td><td>{fin}</td></tr>"
    
    html += "</table>"
    display(HTML(html))

show_clean_table(w_list, u_list, deg_list, fin_list)

In [10]:
def prepare_linear_eqns_i(k, psi, gen_par):
    # step 1: prepare solving f_K_i
    # 定义系数函数
    pattern = build_dict(k, psi)
    coeff_pattern = lambda t_res: prod(t_res) * t_res[psi - 2] * pp^(sigma(t_res[psi-2])) #改完这里要改，长度=k-1
    
    w_list = find_valid_tuples_general(pattern, max_index=k-1)
    u_list = [get_u(pattern, word) for word in w_list]
    deg_list = [compute_inverse_times_vector((1,) + word)[-1] for word in w_list]
    fin_list = [compute_final_vector(w,u) for w, u in zip(w_list,u_list)]

    for word in w_list:
        # 输入你的元组 (w1,w2,w3,w4)
        w_tuple = (1,) + word
        # 计算
        M, M_inv, vec, res = compute_inverse_times_vector(w_tuple)
    
    # step 2: compute f_K_i
    final_res = 0
    for w_tuple, u_tuple in zip(w_list,u_list):
        M = build_weight_matrix((1,) + w_tuple)
        M_inv = M.inverse()
        for perm in generate_sequences(u_tuple):
            r_vec = merge_list(gen_par, perm)
            t_res = compute_inverse_times_vector_with_value(M_inv, r_vec) # t_1, ..., t_(k-1)
            cof_res = coeff_pattern(t_res)
            final_res += cof_res
    final_constant = final_res.coefficients()[0]
    if len(final_res.coefficients()) == 2:
        final_coeff = final_res.coefficients()[-1]
    else:
        final_coeff = 0

    # step 3: prepare coefficient matrix
    component_list = []
    for length in range(2,k):
        component_list += [cal_poly(gen_par, length, deg) * n^(k-deg) for deg in range(length+1, k+1)]

    comp_0 = [p.coefficients()[0] for p in component_list]
    comp_1 = [p.coefficients()[-1] if len(p.coefficients()) == 2 else 0 for p in component_list]

    return final_constant, final_coeff, comp_0, comp_1


In [13]:
#--------------
# 算例
# Basic settings
k = 9
psi = 7 # range from 2 to k
n = 3
# Show process or not
if_count = True
if_print = 0

# 👇 就改这里
if psi > k or psi < 2:
    print("psi range from 2 to k")
    # 不合法，直接跳过后面所有内容
    raise SystemExit("参数错误，停止运行本段")

# 下面你原来的代码完全不用动，不用缩进！
l_pos = 1
l_neg = k - l_pos

size = (k - 2) * (k - 1) / 2
A_coefficient_matrix = []
b_vec = []

for it in range(ceil(size/2)):
    if if_count == True:
        print('=' * 80)
        print(f'{it+1}-th computation starts, there are {ceil(size/2)-it-1} times left')
        
    start_time = time.time()
    
    s_pos = 6 + it
    s_neg = s_pos + n
    P_pos_list = Partitions(s_pos, length = l_pos)
    P_neg_list = Partitions(s_neg, length = l_neg)
    lam_pos = P_pos_list.random_element()
    lam_neg = P_neg_list.random_element()

    gen_par = list(lam_neg) + [-i for i in lam_pos]
    
    final_constant, final_coeff, comp_0, comp_1 = prepare_linear_eqns_i(k, psi, gen_par)
    
    A_coefficient_matrix.append(comp_0)
    A_coefficient_matrix.append(comp_1)
    b_vec.append(final_constant)
    b_vec.append(final_coeff)
    
    end_time = time.time()
    
    if if_count == True:
        print('-' * 80)
        print(f'{it+1}-th computation ends, it takes {end_time - start_time}s')
    if if_print == True:
        print(A_coefficient_matrix)
        print(b_vec)

A_coefficient_matrix = np.array(A_coefficient_matrix[:size])
b_vec = np.array(b_vec[:size])
raw_solution = np.linalg.solve(A_coefficient_matrix, b_vec)
solution = [int(round(x)) for x in raw_solution]
print('=' * 80)
print('raw solution is ', raw_solution)
print('solution after rounding is ')
print(solution)
print('=' * 80)

# 响 3 遍
for _ in range(3):
    os.system('afplay /System/Library/Sounds/Glass.aiff')
    time.sleep(1)

1-th computation starts, there are 13 times left
--------------------------------------------------------------------------------
1-th computation ends, it takes 20134.922868967056s
2-th computation starts, there are 12 times left
--------------------------------------------------------------------------------
2-th computation ends, it takes 7068.003194332123s
3-th computation starts, there are 11 times left
--------------------------------------------------------------------------------
3-th computation ends, it takes 7089.191562175751s
4-th computation starts, there are 10 times left
--------------------------------------------------------------------------------
4-th computation ends, it takes 9450.751451253891s
5-th computation starts, there are 9 times left
--------------------------------------------------------------------------------
5-th computation ends, it takes 8444.96842098236s
6-th computation starts, there are 8 times left
------------------------------------------------

Error: AudioQueueStart failed (-66681)
Error: AudioQueueStart failed (-66681)
Error: AudioQueueStart failed (-66681)


In [ ]:
# test

# guess results
# 1. 定义 sigma 函数：负数返回1，0和正数返回0

# 3. 调用函数并输出结果
par = [-i for i in [-5,-1,-1,2,2]]
'''
poly = calculate_polynomial(par, 3, 4)
print("最终的一元多项式：")
print(poly)
'''
guess_res = (
    24*n^5 + 
    20*n^2*cal_poly(par, 2, 3) - 20*n*cal_poly(par, 2, 4) + 6*cal_poly(par, 2, 5)
    + 10*n*cal_poly(par, 3, 4) - 6*cal_poly(par, 3, 5)
    + 6*cal_poly(par, 4, 5)
)
#2*calculate_polynomial(par, 3, 4) + 3*calculate_polynomial(par, 2, 3) - calculate_polynomial(par, 2, 4)
print("guess result is")
print(guess_res)

In [ ]:
if_print and print('ok')